<a href="https://colab.research.google.com/github/keyssptr/Tugas_PDS_Khayla/blob/main/Regresi_Khayla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/refs/heads/main/dataset_fiskal_97_hipoFull.csv"

# Membaca dataset
df = pd.read_csv(url, sep=None, engine='python')

# Memfilter semua baris yang mengandung teks "Kendari" (tidak peka huruf besar/kecil)
kendari_data = df[df.apply(lambda row: row.astype(str).str.contains('Kendari', case=False).any(), axis=1)]

# Menampilkan hasil
print("Kota Kendari:")
display(kendari_data)

Kota Kendari:


,Kota,Tahun,DBH_PAJAK,DBH_SDA,DBH_Lainnya,DAU_Grant,DAU_Earmark,DAK_Fisik,DAK_Nonfisik,PAD,...,bm_lingkungan,bm_ekonomi,bm_infrastruktur,bm_pariwisata,bm_perlindungan_sosial,KEMISKINAN,PENGANGGURAN,KETIMPANGAN,IPM,PE
836,Kota Kendari,2014,46.11,62.28,26.64,130.73,51.05,77.20,53.32,94.21,...,248.45,128.88,206.46,215.48,100.28,18.12,9.01,0.38,77.56,2.30
837,Kota Kendari,2015,85.07,37.42,34.97,114.10,62.16,86.83,53.52,105.54,...,214.86,76.66,63.63,158.51,94.66,13.65,7.60,0.34,69.77,7.24
838,Kota Kendari,2016,62.67,58.74,43.74,123.07,97.41,87.31,73.29,157.79,...,194.45,141.70,259.11,254.86,134.44,15.39,7.15,0.48,68.92,2.45
839,Kota Kendari,2017,58.93,26.11,25.32,139.33,81.48,80.61,78.39,179.55,...,266.53,74.87,177.98,276.40,154.61,12.20,9.09,0.37,83.17,3.48
840,Kota Kendari,2018,68.36,57.86,41.68,116.25,86.52,71.49,57.22,211.91,...,122.36,241.65,181.75,79.94,177.13,14.62,9.14,0.44,68.92,6.17
841,Kota Kendari,2019,91.56,79.79,34.25,157.09,105.13,91.27,105.53,152.76,...,166.92,291.57,190.40,182.98,169.19,19.30,7.92,0.31,82.67,4.46
842,Kota Kendari,2020,82.38,60.16,32.42,139.29,103.29,57.28,71.81,113.10,...,204.85,292.52,92.70,121.13,117.67,11.24,8.10,0.46,83.63,7.26
843,Kota Kendari,2021,76.30,86.61,53.99,156.57,103.13,117.03,104.15,166.45,...,152.80,266.27,99.42,61.77,57.29,15.70,3.28,0.38,70.16,3.79
844,Kota Kendari,2022,55.29,34.23,44.71,151.44,107.45,72.76,86.43,209.38,...,156.87,200.20,129.70,217.14,201.28,7.51,8.02,0.45,62.22,2.07
845,Kota Kendari,2023,129.99,38.24,39.11,146.91,95.69,108.20,109.86,181.76,...,132.57,225.22,198.37,265.69,148.94,18.67,5.58,0.31,67.51,3.80


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio

# Memaksa Plotly untuk merender grafik interaktif langsung di dalam sel Google Colab
pio.renderers.default = 'colab'

# 1. MEMBACA DATA ASLI (TANPA HAPUS OUTLIER)

df_all = pd.read_csv('https://raw.githubusercontent.com/Alam-A99/DATAAKSI/refs/heads/main/dataset_fiskal_97_hipoFull.csv')
df = df_all[df_all['Kota'] == 'Kota Kendari'].copy()

print(f"Total data Kendari yang dianalisis: {len(df)} baris.\n")

# ==========================================
# 2. MODEL REGRESI & UJI ASUMSI KLASIK
# ==========================================
X = df[['bm_ekonomi', 'bm_perlindungan_sosial']]
Y = df['PENGANGGURAN']
X_sm = sm.add_constant(X)
model = sm.OLS(Y, X_sm).fit()

# Menampilkan Summary OLS
print(model.summary())

print("\n" + "="*50)
print("FUNGSI REGRESI (Data Asli):")
print(f"Y = {model.params['const']:.4f} + {model.params['bm_ekonomi']:.4f}(BM_Ekonomi) - {abs(model.params['bm_perlindungan_sosial']):.4f}(BM_Sosial)")
print("="*50)

print("\nHASIL UJI ASUMSI KLASIK")
# 1. Normalitas
shapiro_stat, shapiro_p = stats.shapiro(model.resid)
print(f"- Normalitas (Shapiro-Wilk) P-Value: {shapiro_p:.4f} (>0.05 = Lolos)")

# 2. Multikolinearitas (VIF)
vif_data = pd.DataFrame()
vif_data["Variabel"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(f"- Multikolinearitas (VIF)            : {vif_data['VIF'].max():.4f} (<10 = Lolos)")

# 3. Heteroskedastisitas
bp_test = sms.het_breuschpagan(model.resid, model.model.exog)
print(f"- Heteroskedastisitas (BP) P-Value   : {bp_test[1]:.4f} (>0.05 = Lolos)")

# 4. Autokorelasi
dw_value = sm.stats.stattools.durbin_watson(model.resid)
print(f"- Autokorelasi (Durbin-Watson)       : {dw_value:.4f} (Mendekati 2 = Lolos)\n")

# ==========================================
# 3. VISUALISASI 3D INTERAKTIF DI COLAB
# ==========================================
# Membuat grid bidang regresi berdasarkan rentang data asli
x_range = np.linspace(df['bm_ekonomi'].min(), df['bm_ekonomi'].max(), 20)
y_range = np.linspace(df['bm_perlindungan_sosial'].min(), df['bm_perlindungan_sosial'].max(), 20)
x_grid, y_grid = np.meshgrid(x_range, y_range)

# Menghitung prediksi Z untuk permukaan bidang
z_grid = model.params['const'] + (model.params['bm_ekonomi'] * x_grid) + (model.params['bm_perlindungan_sosial'] * y_grid)

# Inisialisasi Plotly Figure
fig = go.Figure()

# Menambahkan Bidang Regresi Permukaan
fig.add_trace(go.Surface(
    x=x_grid,
    y=y_grid,
    z=z_grid,
    name='Bidang Regresi',
    colorscale='viridis',
    opacity=0.8,
    showscale=False
))

# Menambahkan Titik Data Aktual Asli
fig.add_trace(go.Scatter3d(
    x=df['bm_ekonomi'],
    y=df['bm_perlindungan_sosial'],
    z=df['PENGANGGURAN'],
    mode='markers',
    name='Data Aktual',
    marker=dict(size=6, color='red', line=dict(color='white', width=1))
))

# Mengatur Tampilan Layout Grafik
fig.update_layout(
    title=f"Plot Bidang Regresi 3D: Pengangguran terhadap BM Ekonomi & BM Sosial (Kendari) | R² = {model.rsquared:.2f}",
    template='plotly_dark',
    scene=dict(
        xaxis_title='BM Ekonomi',
        yaxis_title='BM Perlindungan Sosial',
        zaxis_title='Pengangguran'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Menampilkan kanvas interaktif di bawah sel Colab
fig.show()

Total data Kendari yang dianalisis: 11 baris.

                            OLS Regression Results                            
Dep. Variable:           PENGANGGURAN   R-squared:                       0.379
Model:                            OLS   Adj. R-squared:                  0.223
Method:                 Least Squares   F-statistic:                     2.437
Date:                Wed, 14 Oct 2026   Prob (F-statistic):              0.149
Time:                        14:40:02   Log-Likelihood:                -18.986
No. Observations:                  11   AIC:                             43.97
Df Residuals:                       8   BIC:                             45.17
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------